# 01 — Holon Basics

Demonstrates creating holons, adding layer graphs, querying the
holarchy via SPARQL, and membrane validation.

In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

# ── Create a holon ──
ds.add_holon("urn:holon:vancouver", "Vancouver")

# ── Add multi-source interiors ──
ds.add_interior("urn:holon:vancouver", """
    @prefix muni: <urn:municipal:> .
    <urn:city:vancouver> a muni:CityRecord ;
        muni:officialName "City of Vancouver" ;
        muni:censusPop 675218 ;
        muni:lat 49.2827 ;
        muni:lon -123.1207 .
""", graph_iri="urn:holon:vancouver/interior/municipal")

ds.add_interior("urn:holon:vancouver", """
    @prefix census: <urn:census:> .
    <urn:city:vancouver> census:year 2021 ;
        census:province "British Columbia" .
""", graph_iri="urn:holon:vancouver/interior/census")

# ── Add a boundary ──
ds.add_boundary("urn:holon:vancouver", """
    @prefix muni: <urn:municipal:> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    <urn:shapes:CityShape> a sh:NodeShape ;
        sh:targetClass muni:CityRecord ;
        sh:property [
            sh:path muni:officialName ;
            sh:minCount 1 ;
            sh:severity sh:Violation ;
            sh:message "City must have a name."
        ] .
""")

print(ds.summary())

## Query across multiple interior graphs

In [ ]:
# SPARQL sees both interior graphs
rows = ds.query("""
    PREFIX muni: <urn:municipal:>
    PREFIX census: <urn:census:>
    SELECT ?name ?pop ?year
    WHERE {
        GRAPH ?g1 {
            ?city a muni:CityRecord ;
                muni:officialName ?name ;
                muni:censusPop ?pop .
        }
        GRAPH ?g2 {
            ?city census:year ?year .
        }
    }
""")
for row in rows:
    print(f"  {row['name']}: pop={row['pop']}, census year={row['year']}")

## Membrane validation

In [ ]:
result = ds.validate_membrane("urn:holon:vancouver")
print(result.summary())